# Soft Voting Ensemble

**Модели:** Whisper-small + OpenL3 + XLS-R → взвешенное усреднение вероятностей

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import f1_score
from transformers import WhisperProcessor, WhisperModel
import openl3
import soundfile as sf

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate
from shared.data_utils import build_feature_matrix, load_audio
from shared.results_utils import save_result_csv

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

from transformers import Wav2Vec2Model, Wav2Vec2Processor

In [ ]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_test_split()

idx = np.arange(len(paths_trainval))
idx_tr, idx_val = train_test_split(
    idx, test_size=0.18, stratify=labels_trainval, random_state=config.RANDOM_STATE
)
paths_train,  paths_val  = paths_trainval[idx_tr],  paths_trainval[idx_val]
labels_train, labels_val = labels_trainval[idx_tr], labels_trainval[idx_val]
letters_train, letters_val = letters_trainval[idx_tr], letters_trainval[idx_val]

print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")

In [ ]:
print(f"Device: {DEVICE}")

# --- Whisper ---
W_MODEL_ID = "openai/whisper-small"
w_processor = WhisperProcessor.from_pretrained(W_MODEL_ID)
whisper = WhisperModel.from_pretrained(W_MODEL_ID).to(DEVICE)
whisper.eval()

def extract_whisper(path):
    y, _ = load_audio(path, sr=16000)
    feats = w_processor.feature_extractor(y, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        hs = whisper.encoder(input_features=feats.input_features.to(DEVICE)).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

# --- OpenL3 ---
def extract_openl3(path):
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    emb, _ = openl3.get_audio_embedding(audio, sr, content_type="music",
                                         embedding_size=256, hop_size=0.5, verbose=False)
    return np.concatenate([emb.mean(0), emb.max(0)]).astype(np.float32)

# --- XLS-R ---
xlsr_proc  = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-xls-r-300m")
xlsr_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-xls-r-300m").to(DEVICE)
xlsr_model.eval()

def extract_xlsr(path):
    y, _ = load_audio(path, sr=16000)
    inp = xlsr_proc(y, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        hs = xlsr_model(inp.input_values.to(DEVICE)).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(0), hs.std(0)]).astype(np.float32)

In [ ]:
def train_svm(X_tr, y_tr, letters_tr, X_eval, letters_eval):
    X_tr_full   = np.hstack([X_tr,   letters_tr])
    X_eval_full = np.hstack([X_eval, letters_eval])
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", class_weight="balanced",
                    probability=True, random_state=config.RANDOM_STATE)),
    ])
    grid = GridSearchCV(pipe,
                        {"clf__C": [0.5, 1.0, 5.0, 10.0],
                         "clf__gamma": ["scale", 0.01, 0.1]},
                        cv=3, scoring="f1_macro", refit=True, n_jobs=-1)
    grid.fit(X_tr_full, y_tr)
    proba_eval = grid.best_estimator_.predict_proba(X_eval_full)[:, config.CLASS_BAD]
    return grid.best_estimator_, proba_eval


t0 = time.perf_counter()

print("=== Whisper ===")
X_tr_w  = build_feature_matrix(paths_train, extract_whisper, n_jobs=1)
X_val_w = build_feature_matrix(paths_val,   extract_whisper, n_jobs=1)
X_te_w  = build_feature_matrix(paths_test,  extract_whisper, n_jobs=1)
clf_w, proba_val_w = train_svm(X_tr_w, labels_train, letters_train, X_val_w, letters_val)
proba_te_w = clf_w.predict_proba(np.hstack([X_te_w, letters_test]))[:, config.CLASS_BAD]

print("\n=== OpenL3 ===")
X_tr_o  = build_feature_matrix(paths_train, extract_openl3, n_jobs=4)
X_val_o = build_feature_matrix(paths_val,   extract_openl3, n_jobs=4)
X_te_o  = build_feature_matrix(paths_test,  extract_openl3, n_jobs=4)
clf_o, proba_val_o = train_svm(X_tr_o, labels_train, letters_train, X_val_o, letters_val)
proba_te_o = clf_o.predict_proba(np.hstack([X_te_o, letters_test]))[:, config.CLASS_BAD]

print("\n=== XLS-R ===")
X_tr_x  = build_feature_matrix(paths_train, extract_xlsr, n_jobs=1)
X_val_x = build_feature_matrix(paths_val,   extract_xlsr, n_jobs=1)
X_te_x  = build_feature_matrix(paths_test,  extract_xlsr, n_jobs=1)
clf_x, proba_val_x = train_svm(X_tr_x, labels_train, letters_train, X_val_x, letters_val)
proba_te_x = clf_x.predict_proba(np.hstack([X_te_x, letters_test]))[:, config.CLASS_BAD]

train_time_sec = time.perf_counter() - t0
print(f"\nВремя обучения трёх моделей: {train_time_sec:.1f} с")

# Подбор весов ансамбля на val
best_weights, best_f1_val = (1/3, 1/3, 1/3), -1.0
for w1 in np.arange(0.2, 0.7, 0.1):
    for w2 in np.arange(0.1, 0.5, 0.1):
        w3 = 1.0 - w1 - w2
        if w3 < 0.05:
            continue
        proba_ens = w1 * proba_val_w + w2 * proba_val_o + w3 * proba_val_x
        thr = find_optimal_threshold(labels_val, proba_ens)
        f1 = f1_score(labels_val, (proba_ens >= thr).astype(int),
                      pos_label=config.CLASS_BAD)
        if f1 > best_f1_val:
            best_f1_val = f1
            best_weights = (round(w1, 2), round(w2, 2), round(w3, 2))

w1, w2, w3 = best_weights
print(f"Лучшие веса: Whisper={w1}, OpenL3={w2}, XLS-R={w3}")
print(f"F1-bad на val (ансамбль): {best_f1_val:.4f}")


# Одиночные модели на val для сравнения
for name, proba in [("Whisper", proba_val_w), ("OpenL3", proba_val_o), ("XLS-R", proba_val_x)]:
    thr = find_optimal_threshold(labels_val, proba)
    f1 = f1_score(labels_val, (proba >= thr).astype(int), pos_label=config.CLASS_BAD)
    print(f"  {name}: F1-bad={f1:.4f} (thr={thr:.2f})")

In [ ]:
# Оценка на тесте
proba_ens_test = w1 * proba_te_w + w2 * proba_te_o + w3 * proba_te_x
proba_ens_val  = w1 * proba_val_w + w2 * proba_val_o + w3 * proba_val_x
optimal_threshold = find_optimal_threshold(labels_val, proba_ens_val)
print(f"Оптимальный порог: {optimal_threshold:.2f}")

test_metrics = evaluate(labels_test, proba_ens_test, threshold=optimal_threshold, verbose=True)
pd.DataFrame({
    "path":    paths_test,
    "y_true":  labels_test,
    "y_pred":  (proba_ens_test >= optimal_threshold).astype(int),
    "y_proba": proba_ens_test,
}).to_csv(exp_dir / "test_predictions.csv", index=False)

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_ensemble_soft_voting",
    experiment_name="Soft Voting Ensemble (Whisper + OpenL3 + XLS-R)",
    accuracy=test_metrics["accuracy"],
    f1_macro=test_metrics["f1_macro"],
    f1_bad=test_metrics["f1_bad"],
    roc_auc=test_metrics["roc_auc"],
    precision_bad=test_metrics["precision_bad"],
    recall_bad=test_metrics["recall_bad"],
    threshold=test_metrics["threshold"],
    embed_dim=None,
    embed_dim_note="Ансамбль: единый вектор эмбеддинга отсутствует",
    notes=(
        f"Whisper×{w1} + OpenL3×{w2} + XLS-R×{w3} | "
        f"weights tuned on val by F1-bad | thr={optimal_threshold:.2f}"
    ),
    train_time_sec=train_time_sec,
)
print("Результаты сохранены")
comparison = pd.DataFrame([
    {"Модель": "Whisper SVM (exp_30)",   "F1-bad": 0.722, "ROC-AUC": 0.890},
    {"Модель": "OpenL3 SVM (exp_18)",    "F1-bad": 0.664, "ROC-AUC": 0.817},
    {"Модель": "XLS-R SVM (exp_23)",     "F1-bad": 0.642, "ROC-AUC": 0.833},
    {"Модель": f"Ансамбль ({w1}/{w2}/{w3}) [этот]",
     "F1-bad": test_metrics["f1_bad"],
     "ROC-AUC": test_metrics["roc_auc"]},
])
display(comparison.set_index("Модель").round(4))